In [2]:
import cv2
import pandas as pd
from pathlib import Path

In [3]:
downloaded_videos = Path("external/video-preprocessing/youtube-taichi").glob("*.mp4")
selected_vids = set([video.stem for video in downloaded_videos][:1])
df = pd.read_csv('datasets/ted-metadata.csv')

In [4]:
selected_vids

{'TLZ6W-Nqv1I'}

In [5]:
mask = df['video_id'].str.split('#').apply(lambda x: x[0] in selected_vids)
df = df[mask]

In [6]:
df.to_csv('datasets/ted-metadata-32.csv', index=False)

In [7]:
# df

In [10]:
df['bbox'].iloc[0]

'259-51-716-509'

In [23]:
import numpy as np
def resize_and_pad(frame, target_size=384):
    h, w = frame.shape[:2]
    print(h, w)
    if h > w:
        new_h = target_size
        new_w = int(w * target_size / h)
    else:
        new_w = target_size
        new_h = int(h * target_size / w)
    
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    
    # Create a 384x384 black canvas
    padded = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    
    # Center the resized image
    x_offset = (target_size - new_w) // 2
    y_offset = (target_size - new_h) // 2
    padded[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    
    return padded


In [27]:
from ultralytics import YOLO
# x1, y1, x2, y2 = map(int, df['bbox'].iloc[0].split('-'))
model = YOLO('yolo11n.pt')

video_id = df['video_id'].iloc[0]
parts = video_id.split('#')

video_path = f'external/video-preprocessing/youtube-taichi/{parts[0]}.mp4'
start = int(parts[1])   # Convert milliseconds to seconds
end = int(parts[2]) 

cap = cv2.VideoCapture(video_path)

original_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

start_frame = int(start )
end_frame = int(end )

In [28]:
fps = 10

output_path = f"{video_id}_processed.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, original_fps, (384, 384))
# Set frame position to start
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)


current_frame = start_frame
while current_frame < end_frame and cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Crop the frame
    bbox = model.predict(frame, verbose=False)[0].boxes.xyxy[0]
    x1, y1, x2, y2 = map(int, bbox)
    w = x2 - x1
    h = y2 - y1
    if w > h:
        y1 = max(0, y1 - (w - h) // 2)
        y2 = min(frame.shape[0], y2 + (w - h) // 2)
    else:
        x1 = max(0, x1 - (h - w) // 2)
        x2 = min(frame.shape[1], x2 + (h - w) // 2)
    cropped_frame = frame[y1:y2, x1:x2]
    
    # Resize to 384x384
    resized_frame = cv2.resize(cropped_frame, (384, 384), interpolation=cv2.INTER_LINEAR)
    
    # Write to output video
    out.write(resized_frame)
    
    current_frame += 1

# Release resources
cap.release()
out.release()

In [ ]:
import os
import numpy as np
from ultralytics import YOLO

def is_full_body(bbox, frame_height, min_height_ratio=0.5, min_aspect_ratio=1.5):
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    aspect_ratio = height / max(width, 1)
    height_coverage = height / frame_height
    return aspect_ratio > min_aspect_ratio and height_coverage > min_height_ratio


# Function to resize and pad to 384x384
def resize_and_pad(frame, target_size=384):
    h, w = frame.shape[:2]
    if h > w:
        new_h = target_size
        new_w = int(w * target_size / h)
    else:
        new_w = target_size
        new_h = int(h * target_size / w)
    
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    
    # Create a 384x384 black canvas
    padded = np.zeros((target_size, target_size, 3), dtype=np.uint8)
    
    # Center the resized image
    x_offset = (target_size - new_w) // 2
    y_offset = (target_size - new_h) // 2
    padded[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    
    return padded



In [27]:
# process_video(video_path, 'output', df['video_id'].loc[0], fps, start, end)

In [30]:
video_id = df['video_id'].iloc[0]
fps = 0
start = video_id.split('#')[1]
end = video_id.split('#')[2]

In [ ]:
cap = cv2.VideoCapture(video_path)


original_fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

start_frame = int(start * original_fps)
end_frame = min(int(end * original_fps), total_frames)